# Experiment 2: Implementation of Multiple Linear Regression

**Course:** BCSE0731 — Machine Learning Lab  
**Aim:** Build a Multiple Linear Regression model on the real-world **Diabetes Progression** dataset (`sklearn.datasets.load_diabetes`) and verify feature encoding, scaling, coefficient interpretation, and regression metrics.

### Mathematical Model
$$\hat{y} = \beta_0 + \beta_1 x_1 + \beta_2 x_2 + \dots + \beta_p x_p = X\beta$$

### Step 1: Import Libraries

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.datasets import load_diabetes
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

### Step 2: Load & Explore Real-World Dataset

In [ ]:
diabetes = load_diabetes(as_frame=True)
df = diabetes.frame.copy()
print("Dataset Shape:", df.shape)
print("Features:", list(diabetes.feature_names))
print("Missing values:", df.isnull().sum().sum())
display(df.head())
display(df.describe().T[['mean', 'std', 'min', 'max']])

### Step 3: Correlation Heatmap Inspection

In [ ]:
corr = df.corr()
fig, ax = plt.subplots(figsize=(9, 7))
im = ax.imshow(corr, cmap='coolwarm', vmin=-1, vmax=1)
plt.colorbar(im, ax=ax, fraction=0.046)
ax.set_xticks(range(len(corr.columns)))
ax.set_yticks(range(len(corr.columns)))
ax.set_xticklabels(corr.columns, rotation=45, ha='right')
ax.set_yticklabels(corr.columns)
for i in range(len(corr.columns)):
    for j in range(len(corr.columns)):
        ax.text(j, i, f"{corr.iloc[i, j]:.2f}", ha='center', va='center', color='black', fontsize=8)
ax.set_title('Feature Correlation Matrix (Diabetes Dataset)', fontweight='bold')
plt.tight_layout()
plt.show()

### Step 4: Train-Test Split, Scaling & Model Training

In [ ]:
X = df.drop(columns=['target'])
y = df['target']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

model = LinearRegression()
model.fit(X_train_scaled, y_train)

coef_df = pd.DataFrame({
    'Feature': X.columns,
    'Coefficient': np.round(model.coef_, 3)
}).sort_values(by='Coefficient', key=abs, ascending=False)

print(f"Intercept (beta_0): {model.intercept_:.3f}")
display(coef_df)

### Step 5: Evaluation & Visualizations

In [ ]:
y_pred = model.predict(X_test_scaled)
mae = mean_absolute_error(y_test, y_pred)
rmse = np.sqrt(mean_squared_error(y_test, y_pred))
r2 = r2_score(y_test, y_pred)
n, p = X_test.shape
adj_r2 = 1 - (1 - r2) * (n - 1) / (n - p - 1)

print(f"MAE         : {mae:.3f}")
print(f"RMSE        : {rmse:.3f}")
print(f"R2 Score    : {r2:.4f}")
print(f"Adjusted R2 : {adj_r2:.4f}")

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 5))
ax1.scatter(y_test, y_pred, color='#4F46E5', alpha=0.75, edgecolor='#1E1B4B')
lims = [min(y_test.min(), y_pred.min()), max(y_test.max(), y_pred.max())]
ax1.plot(lims, lims, 'r--', lw=2, label='Ideal Fit (y = y_hat)')
ax1.set_title('Actual vs. Predicted Values', fontweight='bold')
ax1.set_xlabel('Actual Target'); ax1.set_ylabel('Predicted Target')
ax1.legend(); ax1.grid(True, linestyle='--', alpha=0.4)

ax2.barh(coef_df['Feature'][::-1], coef_df['Coefficient'][::-1], color='#0D9488', edgecolor='#1E1B4B')
ax2.axvline(0, color='black', lw=1)
ax2.set_title('Standardized Feature Coefficients', fontweight='bold')
ax2.set_xlabel('Weight Magnitude')
ax2.grid(True, axis='x', linestyle='--', alpha=0.4)

plt.tight_layout()
plt.show()

### Conclusion
- Multiple Linear Regression models the linear effect of all predictors simultaneously.
- Adjusted $R^2$ penalizes unnecessary features, and standardized coefficients reveal that `bmi` and `s5` carry the strongest influence.